# EWC experiments on Colab GPU

Speed test: condition 1 only (1 epoch, 2 SNN time steps), one seed, ANN and SNN.
Code is cloned from GitHub, so **commit and push local changes first**.

In [ ]:
# 1. Confirm this kernel is the Colab T4, not your laptop.
import platform, torch
print("machine:", platform.node(), platform.system())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: select the Colab kernel and a T4 GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the latest code from GitHub and install the one missing package.
import os, sys
REPO = "/content/Dissertation"
if not os.path.exists(REPO):
    !git clone -q https://github.com/ghonilcloud/Dissertation.git {REPO}
%cd {REPO}
!git pull -q
!git log --oneline -1
!pip install -q spikingjelly
sys.path.insert(0, REPO)

In [ ]:
# 3. Stop early if GitHub still has the old code (conditions without ewc_lambda/replay).
import ewc.runEwcExperiments as r
assert "replay" in r.EWC_EXPERIMENT_CONDITIONS[0], "Old code on GitHub: commit and push, then re-run cell 2."
print(r.EWC_EXPERIMENT_CONDITIONS[0])

In [ ]:
# 4. Download CIFAR-10 (data/ is not in git).
from utilities.dataset import get_cifar_datasets
_ = get_cifar_datasets("cifar10", "data", download=True)

In [ ]:
# 5. Speed test: condition 1, seed 42, ANN + SNN, task scenario.
import time
r.EWC_EXPERIMENT_CONDITIONS = r.EWC_EXPERIMENT_CONDITIONS[:1]  # condition 1 only
r.DEFAULT_SEEDS = (42,)                                        # one seed for the test
start = time.perf_counter()
res = r.run_grid(output_path="results/colab_test_condition1.json", progress=True)
wall = time.perf_counter() - start
c = res["conditions"][0]
for m in ("ann", "snn"):
    print(f"{m}: final accuracy {c[m]['mean_final_accuracy']:.3f}, training time {c[m]['training_time_total_seconds']:.1f}s")
print(f"wall time (incl. EWC Fisher + evaluation): {wall:.1f}s")

In [ ]:
# 6. Save the result to your computer (Colab files are deleted when the session ends).
from google.colab import files
files.download("results/colab_test_condition1.json")